# MCP & Tool Integration
The **Model Context Protocol** (MCP) standardizes how LLMs connect to external tools.
Prerequisites: `pip install numpy matplotlib`

📺 **Video Lecture:** [https://youtu.be/DtbQVgjjdSs](https://youtu.be/DtbQVgjjdSs)

In [2]:
import numpy as np
import json
import matplotlib.pyplot as plt

## 1. What Is MCP?
MCP provides a universal protocol for connecting AI models to data sources and tools.
Think of it as a 'USB-C for AI' — one standard connector for everything.

In [3]:
# MCP Architecture: Host → Client → Server
# Let's simulate this architecture

class MCPServer:
    """Simulates an MCP server that exposes tools."""
    def __init__(self, name):
        self.name = name
        self.tools = {}
    
    def register_tool(self, tool_name, func, schema):
        self.tools[tool_name] = {'func': func, 'schema': schema}
    
    def list_tools(self):
        return {name: info['schema'] for name, info in self.tools.items()}
    
    def call_tool(self, tool_name, params):
        if tool_name not in self.tools:
            return {'error': f'Tool {tool_name} not found'}
        return self.tools[tool_name]['func'](params)

# Create a server with tools
server = MCPServer('WeatherServer')
server.register_tool(
    'get_weather',
    lambda p: {'temp': 72, 'condition': 'sunny', 'city': p.get('city', 'unknown')},
    {'type': 'object', 'properties': {'city': {'type': 'string'}}}
)

print('Available tools:', json.dumps(server.list_tools(), indent=2))
print('\nCall result:', server.call_tool('get_weather', {'city': 'San Francisco'}))

Available tools: {
  "get_weather": {
    "type": "object",
    "properties": {
      "city": {
        "type": "string"
      }
    }
  }
}

Call result: {'temp': 72, 'condition': 'sunny', 'city': 'San Francisco'}


## 2. Tool Schema Definition
MCP tools are defined with JSON Schema — the LLM reads these to know what tools exist.

In [4]:
# Example tool schemas (following MCP specification)
tool_schemas = [
    {
        'name': 'read_file',
        'description': 'Read contents of a file',
        'inputSchema': {
            'type': 'object',
            'properties': {
                'path': {'type': 'string', 'description': 'File path to read'},
            },
            'required': ['path']
        }
    },
    {
        'name': 'search_web',
        'description': 'Search the web for information',
        'inputSchema': {
            'type': 'object',
            'properties': {
                'query': {'type': 'string', 'description': 'Search query'},
                'max_results': {'type': 'integer', 'default': 5},
            },
            'required': ['query']
        }
    },
]

print('MCP Tool Definitions:')
for tool in tool_schemas:
    print(f"\n  Tool: {tool['name']}")
    print(f"  Description: {tool['description']}")
    print(f"  Required params: {tool['inputSchema'].get('required', [])}")

MCP Tool Definitions:

  Tool: read_file
  Description: Read contents of a file
  Required params: ['path']

  Tool: search_web
  Description: Search the web for information
  Required params: ['query']


## 3. Client-Server Communication Flow

In [5]:
class MCPClient:
    """Simulates the MCP client that bridges host and server."""
    def __init__(self):
        self.servers = {}
    
    def connect(self, server):
        self.servers[server.name] = server
        print(f'Connected to {server.name}')
    
    def discover_tools(self):
        all_tools = {}
        for srv_name, srv in self.servers.items():
            for tool_name, schema in srv.list_tools().items():
                all_tools[f'{srv_name}/{tool_name}'] = schema
        return all_tools
    
    def invoke(self, server_name, tool_name, params):
        if server_name not in self.servers:
            return {'error': 'Server not found'}
        return self.servers[server_name].call_tool(tool_name, params)

# Simulate the full flow
db_server = MCPServer('DatabaseServer')
db_server.register_tool('query', lambda p: {'rows': 42, 'sql': p.get('sql','')},
                        {'properties': {'sql': {'type': 'string'}}})

client = MCPClient()
client.connect(server)
client.connect(db_server)

print('\nDiscovered tools:', list(client.discover_tools().keys()))
print('\nQuery result:', client.invoke('DatabaseServer', 'query', {'sql': 'SELECT COUNT(*) FROM users'}))

Connected to WeatherServer
Connected to DatabaseServer

Discovered tools: ['WeatherServer/get_weather', 'DatabaseServer/query']

Query result: {'rows': 42, 'sql': 'SELECT COUNT(*) FROM users'}


## 4. Resources and Prompts in MCP
MCP also defines **Resources** (data) and **Prompts** (templates) beyond just tools.

In [6]:
# Resources: structured data the server can expose
class MCPResource:
    def __init__(self, uri, name, mime_type, content):
        self.uri = uri
        self.name = name
        self.mime_type = mime_type
        self.content = content

resources = [
    MCPResource('file:///data/users.csv', 'Users Dataset', 'text/csv', 'id,name\n1,Alice\n2,Bob'),
    MCPResource('db://analytics/metrics', 'Key Metrics', 'application/json', '{"dau": 50000}'),
]

# Prompts: reusable templates
prompts = {
    'summarize': {
        'name': 'summarize',
        'description': 'Summarize provided content',
        'arguments': [{'name': 'content', 'required': True}],
        'template': 'Please summarize the following:\n\n{content}'
    }
}

print('Available Resources:')
for r in resources:
    print(f'  {r.uri} ({r.mime_type})')
print(f'\nAvailable Prompts: {list(prompts.keys())}')

Available Resources:
  file:///data/users.csv (text/csv)
  db://analytics/metrics (application/json)

Available Prompts: ['summarize']


## 5. Comparing Integration Approaches
MCP vs Function Calling vs Plugins — what's different?

In [7]:
import pandas as pd

comparison = pd.DataFrame({
    'Feature': ['Standard Protocol', 'Multi-server', 'Dynamic Discovery',
                'Stateful Sessions', 'Resource Access', 'Provider Agnostic'],
    'MCP':              ['Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes'],
    'Function Calling': ['No',  'No',  'No',  'No',  'No',  'Varies'],
    'Plugins (ChatGPT)':['No',  'Limited', 'Limited', 'No', 'Limited', 'No'],
})
print(comparison.to_string(index=False))

          Feature MCP Function Calling Plugins (ChatGPT)
Standard Protocol Yes               No                No
     Multi-server Yes               No           Limited
Dynamic Discovery Yes               No           Limited
Stateful Sessions Yes               No                No
  Resource Access Yes               No           Limited
Provider Agnostic Yes           Varies                No


## 6. Interview Takeaways
- **MCP** = Model Context Protocol, a standard for AI-tool integration
- Architecture: **Host** (app) → **Client** → **Server** (tools/resources)
- Three primitives: **Tools** (actions), **Resources** (data), **Prompts** (templates)
- Tools are defined with JSON Schema for automatic discovery
- MCP is provider-agnostic — works with any LLM
- Enables composable, reusable integrations across AI applications

---

<small><em>© 2026 AI Nirvana · More Info: https://medium.com/@snigam/a-simple-structured-way-to-prepare-for-ai-ml-interviews-68b2e5830195 · Disclaimer: Provided as is. No liability assumed.</em></small>